In [1]:
import pandas as pd

In [2]:
df_accessions = pd.read_csv('accessions_2007_2022.csv')
df_unemp = pd.read_csv('unemployment_fy2007_2022.csv')

In [3]:
print('Accessions shape:', df_accessions.shape)
print('Unemployment shape:', df_unemp.shape)
print()
print('Accessions years:', sorted(df_accessions.year.unique()))
print('Unemployment years:', sorted(df_unemp.Year.unique()))
print()
print('Accessions states:', df_accessions.state.nunique())
print('Unemployment states:', df_unemp.state.nunique())

Accessions shape: (816, 43)
Unemployment shape: (816, 6)

Accessions years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Unemployment years: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

Accessions states: 51
Unemployment states: 51


In [4]:
df_unemp.rename(columns={'Year': 'year'}, inplace=True)

In [5]:
df_merged = pd.merge(
    df_accessions,
    df_unemp[['state', 'year', 'unemp_rate_mean']],
    on=['state', 'year'],
    how='left'   # keep all accession rows, match unemployment where available
)

In [6]:
print(f'Rows before merge: {len(df_accessions)}')
print(f'Rows after merge:  {len(df_merged)}')
print(f'Unmatched rows:    {df_merged.unemp_rate_mean.isna().sum()}')

Rows before merge: 816
Rows after merge:  816
Unmatched rows:    0


In [7]:
# Check state names match exactly between datasets
accession_states = set(df_accessions.state.unique())
unemp_states = set(df_unemp.state.unique())

only_in_accessions = accession_states - unemp_states
only_in_unemp = unemp_states - accession_states

print('In accessions but not unemployment:', only_in_accessions)
print('In unemployment but not accessions:', only_in_unemp)

In accessions but not unemployment: set()
In unemployment but not accessions: set()


In [8]:
promise_programs = {
    'Tennessee':    2014,
    'Oregon':       2015,
    'Kentucky':     2016,
    'Arkansas':     2017,
    'California':   2017,
    'Hawaii':       2017,
    'Indiana':      2017,
    'Montana':      2017,
    'Nevada':       2017,
    'New York':     2017,
    'Rhode Island': 2017,
    'Maryland':     2018,
    'New Jersey':   2018,
    'Utah':         2019,
    'Washington':   2019,
    'West Virginia':2019,
    'New Mexico':   2020,
    'Connecticut':  2020,
    'Maine':        2022,
}

In [9]:
# Create binary treatment variable
def assign_promise(row):
    state = row['state']
    year = row['year']
    if state in promise_programs:
        return 1 if year >= promise_programs[state] else 0
    return 0

df_merged['promise'] = df_merged.apply(assign_promise, axis=1)

# Also create years since adoption variable for event study
def years_since_promise(row):
    state = row['state']
    year = row['year']
    if state in promise_programs:
        return year - promise_programs[state]
    return None  # never treated states get None

df_merged['years_since_promise'] = df_merged.apply(years_since_promise, axis=1)

In [10]:
df_merged.to_csv('analysis_panel_2007_2022.csv', index=False)
print('Saved analysis_panel_2007_2022.csv')
print(f'Shape: {df_merged.shape}')
print(f'Columns: {list(df_merged.columns)}')

Saved analysis_panel_2007_2022.csv
Shape: (816, 46)
Columns: ['state', 'dod_males', 'dod_males_pct', 'dod_females', 'dod_females_pct', 'dod_total', 'dod_total_pct_x', 'civ_males_pct', 'civ_females_pct', 'civ_total_pct_x', 'year', 'dod_white', 'dod_white_pct', 'dod_black', 'dod_black_pct', 'dod_aian', 'dod_aian_pct', 'dod_asian', 'dod_asian_pct', 'dod_nhpi', 'dod_nhpi_pct', 'dod_two_or_more', 'dod_two_pct', 'dod_unknown', 'dod_unknown_pct', 'dod_total_race', 'dod_total_pct_y', 'civ_white', 'civ_white_pct', 'civ_black', 'civ_black_pct', 'civ_aian', 'civ_aian_pct', 'civ_asian', 'civ_asian_pct', 'civ_nhpi', 'civ_nhpi_pct', 'civ_two_or_more', 'civ_two_pct', 'civ_total', 'civ_total_pct_y', 'civ_males', 'civ_females', 'unemp_rate_mean', 'promise', 'years_since_promise']
